# 🏦 WealthWise AI — Financial Health Score Engine
### Notebook 04: Composite Scoring, Wellness Classification & Personalised Insights

---

## 📋 Table of Contents

| Section | Topic |
|---------|-------|
| 1  | Project Introduction |
| 2  | Import Libraries |
| 3  | Load Dataset |
| 4  | Data Validation |
| 5  | Normalize Scoring Components |
| 6  | Calculate Financial Health Score |
| 7  | Health Categories |
| 8  | Strength Analysis |
| 9  | Weakness Analysis |
| 10 | Personalised Insights |
| 11 | Exploratory Analysis |
| 12 | Top and Bottom Users |
| 13 | Export Dataset |
| 14 | Business Interpretation |

---

> **Author:** WealthWise AI Engineering Team  
> **Version:** 1.0.0  
> **Date:** June 2026  
> **Status:** Production-Ready  
> **Input:** `features_dataset.csv` (from Notebook 02)  
> **Outputs:** `health_score_dataset.csv`, `health_score_dataset.xlsx`

---

# Section 1 — Project Introduction

---

## 🚀 WealthWise AI: Platform Status

| Module | Notebook | Status |
|--------|----------|--------|
| 🧹 Data Cleaning | 01_data_cleaning | ✅ Complete |
| 🔧 Feature Engineering | 02_feature_engineering | ✅ Complete |
| 🤖 Risk Profile Classification | 03_risk_profile_model | ✅ Complete |
| 🩺 Financial Health Score | **04_financial_health_score** | 🔄 **This Notebook** |
| 📈 Portfolio Recommendation | 05_portfolio_recommendation | 🔜 Coming Next |
| 🧠 AI Financial Coach | 06_ai_financial_coach | 🔜 Coming Next |

---

## 🩺 What is a Financial Health Score?

A **Financial Health Score** is a single composite number (0–100) that summarises the overall financial wellness of an account holder based on their observed banking behaviour. It is analogous to:

- A **credit score** (CIBIL/FICO) but focused on *behavioural wellness*, not creditworthiness
- A **fitness score** that quantifies multiple health dimensions into one meaningful number
- A **NPS (Net Promoter Score)** for personal finance — a north-star metric for the user

### Why a Single Score?

Users cannot simultaneously track 37 financial features. A single, interpretable score:
- Provides an **immediate benchmark** ("I am at 72/100 this month")
- Enables **trend tracking** over time ("My score improved by 8 points")
- Drives **actionable focus** ("I need to improve my savings rate")

---

## 📊 Why It Matters

| Problem | How Health Score Solves It |
|---------|---------------------------|
| Financial literacy gap | Translates complex metrics into one intuitive number |
| Lack of self-awareness | Surfaces strengths and weaknesses automatically |
| Inconsistent behaviour | Month-over-month score comparison drives improvement |
| Generic advice | Score tier enables personalised coaching from AI |
| No early warnings | Poor score flags users before they face a financial crisis |

---

## 📝 Scoring Methodology

The WealthWise AI Health Score is a **weighted composite** of three orthogonal financial dimensions:

```
Health Score = (0.50 × Savings Rate Score)
             + (0.25 × Income Consistency Score)
             + (0.25 × Expense Stability Score)
```

| Component | Weight | Rationale |
|-----------|--------|-----------|
| Savings Rate Score | 50% | Primary indicator of financial discipline and future security |
| Income Consistency Score | 25% | Stable income = predictable planning capacity |
| Expense Stability Score | 25% | Controlled spending = sustainable financial lifestyle |

**Weight Justification:**  
Savings rate carries the highest weight because it is the single most actionable and predictive metric for long-term financial health, as validated by research from SEBI, AMFI, and international personal finance literature.

---

## 🎯 Health Score Tiers

| Score Range | Category | Meaning |
|-------------|----------|---------|
| 80 – 100 | 🟢 **Excellent** | Financially strong, consistent saver, minimal risk |
| 60 – 79 | 🟡 **Good** | Healthy finances with room for optimisation |
| 40 – 59 | 🟠 **Fair** | Moderate concerns, needs attention and coaching |
| 0 – 39 | 🔴 **Poor** | Significant financial stress, requires urgent action |

---

# Section 2 — Import Libraries

---

In [1]:
# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================

# Standard library
import os
import warnings

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Suppress non-critical warnings ──────────────────────────────────────────
warnings.filterwarnings('ignore')

# ── Display settings ─────────────────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 160)

# ── Matplotlib / Seaborn theme ───────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
FIGURE_DPI  = 120
FIGURE_SIZE = (14, 5)

# ── WealthWise AI Brand Palette ───────────────────────────────────────────────
HEALTH_COLORS = {
    'Excellent': '#27AE60',   # Rich green
    'Good':      '#F39C12',   # Warm amber
    'Fair':      '#E67E22',   # Deep orange
    'Poor':      '#C0392B',   # Crimson red
}

BRAND = {
    'income':   '#2ECC71',
    'expense':  '#E74C3C',
    'savings':  '#3498DB',
    'balance':  '#9B59B6',
    'score':    '#1ABC9C',
    'dark':     '#2C3E50',
    'neutral':  '#95A5A6',
}

# ── Scoring Configuration — single source of truth ──────────────────────────
SCORE_WEIGHTS = {
    'savings_rate_score':         0.50,
    'income_consistency_score_n': 0.25,
    'expense_stability_score_n':  0.25,
}

# Savings rate thresholds (percentage scale)
SAVINGS_RATE_FLOOR = 0.0     # Below this: score = 0
SAVINGS_RATE_CAP   = 60.0    # Above this: score = 100

# Component score thresholds for strengths/weaknesses
STRENGTH_THRESHOLD = 65.0    # Score >= threshold → strength
WEAKNESS_THRESHOLD = 40.0    # Score <  threshold → weakness

print("\u2705 Libraries loaded successfully.")
print(f"   pandas  : {pd.__version__}")
print(f"   numpy   : {np.__version__}")
print(f"   seaborn : {sns.__version__}")
print()
print("Scoring Weights:")
for component, weight in SCORE_WEIGHTS.items():
    print(f"  {component:<35s}: {weight*100:.0f}%")

✅ Libraries loaded successfully.
   pandas  : 2.3.3
   numpy   : 1.23.5
   seaborn : 0.13.2

Scoring Weights:
  savings_rate_score                 : 50%
  income_consistency_score_n         : 25%
  expense_stability_score_n          : 25%


---

# Section 3 — Load Dataset

---

We load `features_dataset.csv` produced by Notebook 02. This file contains one row per `account_number × month_year` with 37 engineered financial features.

In [ ]:
# =============================================================================
# SECTION 3: LOAD DATASET
# =============================================================================

def load_features_dataset(filename: str = 'features_dataset.csv') -> pd.DataFrame:
    """
    Load the features dataset produced by Notebook 02.

    Search order:
      1. Same directory as this notebook
      2. data/processed/ subdirectory

    Parameters
    ----------
    filename : str
        Name of the features CSV file.

    Returns
    -------
    pd.DataFrame
        Loaded features dataset.

    Raises
    ------
    FileNotFoundError
        If the file cannot be located in any search path.
    """
    notebook_dir   = os.path.dirname(os.path.abspath('__file__'))
    candidate_dirs = [
        notebook_dir,
        os.path.join(notebook_dir, 'data', 'processed'),
    ]

    for directory in candidate_dirs:
        filepath = os.path.join(directory, filename)
        if os.path.exists(filepath):
            print(f"  📂 Found : {filepath}")
            return pd.read_csv(filepath, low_memory=False)

    raise FileNotFoundError(
        f"'{filename}' not found.\n"
        f"Searched:\n" + "\n".join(f"  - {d}" for d in candidate_dirs)
        + "\n\nPlease run Notebook 02 (02_feature_engineering.ipynb) first."
    )


# ── Load ─────────────────────────────────────────────────────────────────────
print("📥 Loading features_dataset.csv ...\n")
df = load_features_dataset('features_dataset.csv')

# ── Normalise dtypes ──────────────────────────────────────────────────────────
df['account_number'] = df['account_number'].astype(str)
df['month_year']     = df['month_year'].astype(str)

print(f"\n✅ Dataset loaded successfully.")
print(f"   Shape   : {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"   Accounts: {df['account_number'].nunique()} unique")
print(f"   Periods : {df['month_year'].nunique()} unique months")
print(f"   Columns : {list(df.columns)}")

In [ ]:
# ── Display Sample Rows & Info ────────────────────────────────────────────────
print("=" * 70)
print("  Sample Records (first 5 rows)")
print("=" * 70)
display(df.head())

print("\n" + "=" * 70)
print("  DataFrame Info")
print("=" * 70)
df.info()

---

# Section 4 — Data Validation

---

Before scoring, we validate the three core scoring columns are present, numeric, and within expected ranges. Any data quality issues are surfaced and handled defensively.

In [ ]:
# =============================================================================
# SECTION 4: DATA VALIDATION
# =============================================================================

# ── Required scoring columns ──────────────────────────────────────────────────
REQUIRED_SCORE_COLS = [
    'savings_rate',
    'income_consistency_score',
    'expense_stability_score',
]


def validate_dataset(dataframe: pd.DataFrame) -> None:
    """
    Run comprehensive data quality checks.

    Checks:
    - Required scoring columns are present
    - Missing value counts per column
    - Duplicate row detection
    - Data type classification
    - Value range audit for scoring columns

    Parameters
    ----------
    dataframe : pd.DataFrame
        The features dataset to validate.
    """
    print("=" * 60)
    print("  1) Required Scoring Columns")
    print("=" * 60)
    for col in REQUIRED_SCORE_COLS:
        status = "✅" if col in dataframe.columns else "❌ MISSING"
        print(f"  {status}  {col}")
    missing_req = [c for c in REQUIRED_SCORE_COLS if c not in dataframe.columns]
    if missing_req:
        raise ValueError(
            f"Required scoring columns missing: {missing_req}. "
            "Ensure Notebook 02 was run correctly."
        )

    print("\n" + "=" * 60)
    print("  2) Missing Values")
    print("=" * 60)
    missing = dataframe.isnull().sum()
    missing_cols = missing[missing > 0]
    if missing_cols.empty:
        print("  ✅ No missing values detected.")
    else:
        pct = (missing_cols / len(dataframe) * 100).round(2)
        missing_report = pd.DataFrame({'Count': missing_cols, 'Pct %': pct})
        print(f"  ⚠️  {len(missing_cols)} column(s) have missing values:")
        display(missing_report)

    print("\n" + "=" * 60)
    print("  3) Duplicate Rows")
    print("=" * 60)
    n_dup = dataframe.duplicated().sum()
    if n_dup == 0:
        print("  ✅ No duplicate rows detected.")
    else:
        print(f"  ⚠️  {n_dup:,} duplicate rows found — will be dropped.")
        dataframe.drop_duplicates(inplace=True)
        dataframe.reset_index(drop=True, inplace=True)

    print("\n" + "=" * 60)
    print("  4) Data Type Distribution")
    print("=" * 60)
    for dtype, count in dataframe.dtypes.value_counts().items():
        print(f"  {str(dtype):15s}: {count} column(s)")

    print("\n" + "=" * 60)
    print("  5) Scoring Column Value Ranges")
    print("=" * 60)
    print(f"  {'Column':<30s} {'Min':>12s} {'Max':>12s} {'Mean':>12s} {'Nulls':>8s}")
    print(f"  {'-'*76}")
    for col in REQUIRED_SCORE_COLS:
        col_data = dataframe[col]
        print(
            f"  {col:<30s} "
            f"{col_data.min():>12,.4f} "
            f"{col_data.max():>12,.4f} "
            f"{col_data.mean():>12,.4f} "
            f"{col_data.isnull().sum():>8d}"
        )

    print("\n  ✅ Validation complete.")


print("🔍 Running Data Validation...\n")
validate_dataset(df)

In [ ]:
# ── Summary Statistics ────────────────────────────────────────────────────────
print("📊 Summary Statistics for Numeric Features\n")

numeric_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
summary_stats = df[numeric_cols].describe().T
summary_stats['cv_%'] = (
    summary_stats['std'] / summary_stats['mean'].abs() * 100
).round(2)

# Highlight the three scoring columns
score_cols_mask = summary_stats.index.isin(REQUIRED_SCORE_COLS)
display(
    summary_stats
    .style
    .background_gradient(cmap='Blues', subset=['mean', 'std'])
    .set_caption('Summary Statistics — Numeric Features')
)

print(f"\n✅ Statistics generated for {len(numeric_cols)} numeric features.")

---

# Section 5 — Normalize Scoring Components

---

All three components must be normalised to a **0–100 scale** before weighting. We use domain-capped min-max normalisation rather than dataset-relative normalisation, so scores remain consistent as new data arrives.

| Component | Raw Range | Normalisation Method |
|-----------|-----------|---------------------|
| `savings_rate` | −200% to +77% | Clamped to [0, 60], then scaled to [0, 100] |
| `income_consistency_score` | 0 to 91.3 | Clamped to [0, 100], direct use |
| `expense_stability_score` | 0 to 97.8 | Clamped to [0, 100], direct use |

In [ ]:
# =============================================================================
# SECTION 5: NORMALIZE SCORING COMPONENTS
# =============================================================================

def normalise_savings_rate(
    series: pd.Series,
    floor: float = SAVINGS_RATE_FLOOR,
    cap: float   = SAVINGS_RATE_CAP,
) -> pd.Series:
    """
    Normalise savings_rate to a 0-100 score.

    Strategy
    --------
    - Negative savings rates (overspending) are clamped to 0 (score = 0)
    - Savings rates above `cap` are clamped to `cap` (score = 100)
    - Linear scaling between floor and cap

    This domain-capped approach ensures the score is stable across datasets
    and not influenced by outliers in any given month.

    Parameters
    ----------
    series : pd.Series   Raw savings_rate column (percentage scale).
    floor  : float       Value mapped to score 0.   Default: 0.0
    cap    : float       Value mapped to score 100. Default: 60.0

    Returns
    -------
    pd.Series  Normalised score in [0, 100].
    """
    clipped = series.clip(lower=floor, upper=cap)
    normalised = ((clipped - floor) / (cap - floor)) * 100
    return normalised.clip(0, 100).round(4)


def normalise_to_100(
    series: pd.Series,
    floor: float = 0.0,
    cap: float   = 100.0,
) -> pd.Series:
    """
    Normalise a score-like column to a strict 0-100 range.

    Used for income_consistency_score and expense_stability_score
    which are already on a 0-100 scale but may occasionally exceed
    bounds due to floating-point computation in Notebook 02.

    Parameters
    ----------
    series : pd.Series  Input score column.
    floor  : float      Lower bound. Default: 0.0
    cap    : float      Upper bound. Default: 100.0

    Returns
    -------
    pd.Series  Clamped score in [0, 100].
    """
    return series.clip(lower=floor, upper=cap).round(4)


# ── Apply normalisation ───────────────────────────────────────────────────────
print("🔧 Normalising scoring components...\n")

# Component 1: Savings Rate Score
df['savings_rate_score'] = normalise_savings_rate(
    df['savings_rate'],
    floor=SAVINGS_RATE_FLOOR,
    cap=SAVINGS_RATE_CAP,
)

# Component 2: Income Consistency Score (already ~0-100 scale)
df['income_consistency_score_n'] = normalise_to_100(df['income_consistency_score'])

# Component 3: Expense Stability Score (already ~0-100 scale)
df['expense_stability_score_n'] = normalise_to_100(df['expense_stability_score'])

# ── Validation: confirm all components are within [0, 100] ───────────────────
SCORE_COMPONENT_COLS = [
    'savings_rate_score',
    'income_consistency_score_n',
    'expense_stability_score_n',
]

print(f"  {'Component':<35s} {'Min':>8s} {'Max':>8s} {'Mean':>8s} {'Median':>8s}")
print(f"  {'-'*72}")
for col in SCORE_COMPONENT_COLS:
    s = df[col]
    print(
        f"  {col:<35s} "
        f"{s.min():>8.2f} "
        f"{s.max():>8.2f} "
        f"{s.mean():>8.2f} "
        f"{s.median():>8.2f}"
    )

# Boundary assertions
for col in SCORE_COMPONENT_COLS:
    assert df[col].between(0, 100).all(), f"Out-of-range values in {col}!"

print("\n  ✅ All component scores are within [0, 100].")

In [ ]:
# ── Visualise Component Score Distributions ───────────────────────────────────
COMPONENT_LABELS = [
    ('savings_rate_score',         'Savings Rate Score\n(weight: 50%)',      BRAND['savings']),
    ('income_consistency_score_n', 'Income Consistency Score\n(weight: 25%)', BRAND['income']),
    ('expense_stability_score_n',  'Expense Stability Score\n(weight: 25%)', BRAND['balance']),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=FIGURE_DPI)
fig.suptitle(
    'Normalised Scoring Component Distributions (0-100 Scale)',
    fontsize=14, fontweight='bold', y=1.02
)

for ax, (col, label, color) in zip(axes, COMPONENT_LABELS):
    ax.hist(
        df[col].dropna(), bins=25,
        color=color, alpha=0.80,
        edgecolor='white', linewidth=0.5,
    )
    mean_val   = df[col].mean()
    median_val = df[col].median()
    ax.axvline(mean_val,   color='#E74C3C', linestyle='--', linewidth=1.6,
               label=f'Mean: {mean_val:.1f}')
    ax.axvline(median_val, color='#2C3E50', linestyle=':',  linewidth=1.6,
               label=f'Median: {median_val:.1f}')
    ax.set_title(label, fontsize=11, fontweight='bold', pad=10)
    ax.set_xlabel('Score (0 - 100)', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_xlim(0, 100)
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('component_distributions.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Chart saved: component_distributions.png")

---

# Section 6 — Calculate Financial Health Score

---

## Formula

```
Health Score = (0.50 x Savings Rate Score)
             + (0.25 x Income Consistency Score)
             + (0.25 x Expense Stability Score)
```

The result is bounded to **[0, 100]** and rounded to 2 decimal places.

In [ ]:
# =============================================================================
# SECTION 6: CALCULATE FINANCIAL HEALTH SCORE
# =============================================================================

def calculate_health_score(
    row: pd.Series,
    weights: dict = SCORE_WEIGHTS,
) -> float:
    """
    Compute the Financial Health Score for a single account-month record.

    Formula
    -------
    Health Score = sum(weight_i * component_score_i)

    All component scores are expected to be in [0, 100] before calling.
    The result is clipped to [0, 100] as a safety guarantee.

    Parameters
    ----------
    row     : pd.Series  Single row of the features DataFrame.
    weights : dict       Component column -> weight mapping.

    Returns
    -------
    float  Health score in [0, 100], rounded to 2 decimal places.
    """
    raw_score = sum(
        weight * row.get(component, 0.0)
        for component, weight in weights.items()
    )
    return round(float(np.clip(raw_score, 0.0, 100.0)), 2)


# ── Vectorised computation (faster for large datasets) ───────────────────────
df['health_score'] = (
    SCORE_WEIGHTS['savings_rate_score']         * df['savings_rate_score']
    + SCORE_WEIGHTS['income_consistency_score_n'] * df['income_consistency_score_n']
    + SCORE_WEIGHTS['expense_stability_score_n']  * df['expense_stability_score_n']
).clip(0, 100).round(2)

# ── Audit ─────────────────────────────────────────────────────────────────────
hs = df['health_score']
assert hs.between(0, 100).all(), "Health score out of [0, 100] bounds!"

print("✅ Financial Health Score calculated successfully.")
print()
print("  Health Score Distribution Summary:")
print(f"  Min     : {hs.min():.2f}")
print(f"  Max     : {hs.max():.2f}")
print(f"  Mean    : {hs.mean():.2f}")
print(f"  Median  : {hs.median():.2f}")
print(f"  Std Dev : {hs.std():.2f}")
print(f"  Records : {len(hs):,}")

In [ ]:
# ── Verify weighting arithmetic with a sample record ─────────────────────────
print("Sample Weighting Verification (first 3 records):")
print("-" * 80)
verify_cols = [
    'account_number', 'month_year',
    'savings_rate_score', 'income_consistency_score_n',
    'expense_stability_score_n', 'health_score'
]
sample = df[verify_cols].head(3).copy()

# Recompute manually to verify
sample['manual_check'] = (
    0.50 * sample['savings_rate_score']
    + 0.25 * sample['income_consistency_score_n']
    + 0.25 * sample['expense_stability_score_n']
).round(2)

sample['match'] = sample['health_score'] == sample['manual_check']
display(sample)

print(f"\n✅ All manual checks match: {sample['match'].all()}")

---

# Section 7 — Health Categories

---

We map each continuous health score into a discrete wellness category to enable intuitive user communication and UI badge display.

| Score Range | Category | UI Badge |
|-------------|----------|----------|
| 80 – 100 | Excellent | 🟢 Green |
| 60 – 79 | Good | 🟡 Amber |
| 40 – 59 | Fair | 🟠 Orange |
| 0 – 39 | Poor | 🔴 Red |

In [ ]:
# =============================================================================
# SECTION 7: HEALTH CATEGORIES
# =============================================================================

# Category boundary constants
CATEGORY_BINS   = [0, 40, 60, 80, 100]
CATEGORY_LABELS = ['Poor', 'Fair', 'Good', 'Excellent']


def assign_health_category(score: float) -> str:
    """
    Map a health score to a wellness category.

    Parameters
    ----------
    score : float  Health score in [0, 100].

    Returns
    -------
    str  One of: 'Excellent', 'Good', 'Fair', 'Poor'.
    """
    if score >= 80:
        return 'Excellent'
    elif score >= 60:
        return 'Good'
    elif score >= 40:
        return 'Fair'
    else:
        return 'Poor'


# ── Vectorised with pd.cut for efficiency ────────────────────────────────────
df['health_category'] = pd.cut(
    df['health_score'],
    bins=CATEGORY_BINS,
    labels=CATEGORY_LABELS,
    include_lowest=True,
).astype(str)

# ── Distribution Report ───────────────────────────────────────────────────────
cat_counts = df['health_category'].value_counts()
cat_pct    = df['health_category'].value_counts(normalize=True).mul(100).round(2)

dist_df = pd.DataFrame({
    'Count': cat_counts,
    'Percentage': cat_pct.map('{:.2f}%'.format),
})

print("🎯 Health Category Distribution")
print("=" * 45)
display(dist_df)

# ── Per-category score statistics ────────────────────────────────────────────
print("\n📊 Score Statistics by Health Category:")
cat_stats = df.groupby('health_category')['health_score'].agg(
    ['count', 'min', 'mean', 'max', 'std']
).round(2)
cat_stats.columns = ['Count', 'Min', 'Mean', 'Max', 'Std Dev']
display(cat_stats)

In [ ]:
# ── Category Distribution Chart ───────────────────────────────────────────────
ordered_cats   = [c for c in ['Poor', 'Fair', 'Good', 'Excellent'] if c in cat_counts.index]
bar_colors     = [HEALTH_COLORS[c] for c in ordered_cats]
ordered_counts = [cat_counts.get(c, 0) for c in ordered_cats]
ordered_pcts   = [cat_pct.get(c, 0)    for c in ordered_cats]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=FIGURE_DPI)
fig.suptitle(
    'WealthWise AI — Financial Health Category Distribution',
    fontsize=14, fontweight='bold', y=1.02
)

# Bar chart
ax1 = axes[0]
bars = ax1.bar(
    ordered_cats, ordered_counts,
    color=bar_colors, edgecolor='white', linewidth=1.5, width=0.55
)
for bar, cnt, pct in zip(bars, ordered_counts, ordered_pcts):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(ordered_counts) * 0.01,
        f'{cnt:,}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
ax1.set_title('Count by Health Category', fontsize=12, fontweight='bold', pad=12)
ax1.set_xlabel('Health Category', fontsize=11)
ax1.set_ylabel('Number of Records', fontsize=11)
ax1.spines[['top', 'right']].set_visible(False)

# Donut chart
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    ordered_counts, labels=None, colors=bar_colors,
    autopct='%1.1f%%', startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    pctdistance=0.78,
)
# Draw inner circle for donut effect
inner_circle = plt.Circle((0, 0), 0.50, color='white')
ax2.add_patch(inner_circle)
ax2.text(0, 0, f'{df["health_score"].mean():.0f}\nAvg\nScore',
         ha='center', va='center', fontsize=13, fontweight='bold', color=BRAND['dark'])
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')
    at.set_color('white')
patches = [mpatches.Patch(color=HEALTH_COLORS[c], label=c) for c in ordered_cats]
ax2.legend(handles=patches, loc='lower center',
           bbox_to_anchor=(0.5, -0.12), ncol=4, fontsize=10)
ax2.set_title('Proportion by Health Category', fontsize=12, fontweight='bold', pad=12)

plt.tight_layout()
plt.savefig('health_category_distribution.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Chart saved: health_category_distribution.png")

---

# Section 8 — Strength Analysis

---

Strengths are automatically identified when a normalised component score meets or exceeds the **strength threshold (65/100)**. A user may have multiple strengths simultaneously.

| Condition | Strength Label |
|-----------|----------------|
| `savings_rate_score >= 65` | Strong Savings Rate |
| `income_consistency_score_n >= 65` | Consistent Income |
| `expense_stability_score_n >= 65` | Stable Expenses |

In [ ]:
# =============================================================================
# SECTION 8: STRENGTH ANALYSIS
# =============================================================================

# Strength rules: (column, threshold, label)
STRENGTH_RULES = [
    ('savings_rate_score',         STRENGTH_THRESHOLD, 'Strong Savings Rate'),
    ('income_consistency_score_n', STRENGTH_THRESHOLD, 'Consistent Income'),
    ('expense_stability_score_n',  STRENGTH_THRESHOLD, 'Stable Expenses'),
]


def identify_strengths(row: pd.Series, rules: list) -> str:
    """
    Identify financial strengths for a single account-month record.

    A strength is awarded when a component score meets or exceeds
    the defined threshold. Multiple strengths are pipe-separated.

    Parameters
    ----------
    row   : pd.Series  Single row with normalised score columns.
    rules : list       List of (column, threshold, label) tuples.

    Returns
    -------
    str  Pipe-separated list of strengths, or 'None Identified'.
    """
    strengths = [
        label
        for col, threshold, label in rules
        if row.get(col, 0.0) >= threshold
    ]
    return ' | '.join(strengths) if strengths else 'None Identified'


# ── Apply to full dataset ─────────────────────────────────────────────────────
df['strengths'] = df.apply(
    lambda row: identify_strengths(row, STRENGTH_RULES), axis=1
)

# ── Distribution ──────────────────────────────────────────────────────────────
print("💪 Strength Analysis Results")
print("=" * 55)

strength_counts = df['strengths'].value_counts()
print(f"\n  Unique strength combinations: {len(strength_counts)}")
print(f"  Top strength combinations:")
display(strength_counts.head(10).to_frame('Count'))

# Per-component strength rate
print("\n  Individual Component Strength Rates:")
for col, threshold, label in STRENGTH_RULES:
    n = (df[col] >= threshold).sum()
    pct = n / len(df) * 100
    print(f"  {label:<30s}: {n:>5,} records ({pct:>6.1f}%)")

---

# Section 9 — Weakness Analysis

---

Weaknesses are flagged when a normalised component score falls **below the weakness threshold (40/100)**. A user may have multiple weaknesses, each of which becomes an actionable coaching target.

| Condition | Weakness Label |
|-----------|----------------|
| `savings_rate_score < 40` | Low Savings Rate |
| `income_consistency_score_n < 40` | Unstable Income |
| `expense_stability_score_n < 40` | High Expense Variability |

In [ ]:
# =============================================================================
# SECTION 9: WEAKNESS ANALYSIS
# =============================================================================

# Weakness rules: (column, threshold, label)
WEAKNESS_RULES = [
    ('savings_rate_score',         WEAKNESS_THRESHOLD, 'Low Savings Rate'),
    ('income_consistency_score_n', WEAKNESS_THRESHOLD, 'Unstable Income'),
    ('expense_stability_score_n',  WEAKNESS_THRESHOLD, 'High Expense Variability'),
]


def identify_weaknesses(row: pd.Series, rules: list) -> str:
    """
    Identify financial weaknesses for a single account-month record.

    A weakness is flagged when a component score falls below
    the defined threshold. Multiple weaknesses are pipe-separated.

    Parameters
    ----------
    row   : pd.Series  Single row with normalised score columns.
    rules : list       List of (column, threshold, label) tuples.

    Returns
    -------
    str  Pipe-separated list of weaknesses, or 'None Identified'.
    """
    weaknesses = [
        label
        for col, threshold, label in rules
        if row.get(col, 0.0) < threshold
    ]
    return ' | '.join(weaknesses) if weaknesses else 'None Identified'


# ── Apply to full dataset ─────────────────────────────────────────────────────
df['weaknesses'] = df.apply(
    lambda row: identify_weaknesses(row, WEAKNESS_RULES), axis=1
)

# ── Distribution ──────────────────────────────────────────────────────────────
print("⚠️  Weakness Analysis Results")
print("=" * 55)

weakness_counts = df['weaknesses'].value_counts()
print(f"\n  Unique weakness combinations : {len(weakness_counts)}")
print(f"  Top weakness combinations:")
display(weakness_counts.head(10).to_frame('Count'))

# Per-component weakness rate
print("\n  Individual Component Weakness Rates:")
for col, threshold, label in WEAKNESS_RULES:
    n = (df[col] < threshold).sum()
    pct = n / len(df) * 100
    print(f"  {label:<35s}: {n:>5,} records ({pct:>6.1f}%)")

In [ ]:
# ── Strengths vs Weaknesses: Component Breakdown Chart ───────────────────────
component_labels_short = ['Savings Rate', 'Income Consistency', 'Expense Stability']
strength_cols = [col for col, _, _ in STRENGTH_RULES]
weakness_cols = [col for col, _, _ in WEAKNESS_RULES]

strength_pcts = [(df[c] >= STRENGTH_THRESHOLD).mean() * 100 for c in strength_cols]
weakness_pcts = [(df[c] <  WEAKNESS_THRESHOLD).mean() * 100 for c in weakness_cols]

x = np.arange(len(component_labels_short))
w = 0.33

fig, ax = plt.subplots(figsize=(12, 5), dpi=FIGURE_DPI)
bars_s = ax.bar(x - w/2, strength_pcts, w,
                label=f'Strength (score >= {STRENGTH_THRESHOLD:.0f})',
                color='#27AE60', edgecolor='white', linewidth=1.2)
bars_w = ax.bar(x + w/2, weakness_pcts, w,
                label=f'Weakness (score < {WEAKNESS_THRESHOLD:.0f})',
                color='#C0392B', edgecolor='white', linewidth=1.2)

for bar in list(bars_s) + list(bars_w):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{bar.get_height():.1f}%',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax.set_xticks(x)
ax.set_xticklabels(component_labels_short, fontsize=12)
ax.set_ylabel('% of Records', fontsize=11)
ax.set_title(
    'Component-Level Strength vs Weakness Rate',
    fontsize=13, fontweight='bold', pad=14
)
ax.set_ylim(0, 110)
ax.legend(fontsize=11)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('strengths_weaknesses.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Chart saved: strengths_weaknesses.png")

---

# Section 10 — Personalised Insights

---

Each account-month record receives a **plain-English insight** generated via deterministic rule-based logic. These insights are:

- **Contextually accurate** — driven by the actual score values
- **Actionable** — suggest a specific improvement path
- **Concise** — suitable for mobile dashboard display
- **FastAPI-ready** — the function is importable as a service module

In [ ]:
# =============================================================================
# SECTION 10: PERSONALISED INSIGHTS
# =============================================================================

# Insight generation thresholds
HIGH_THRESHOLD = 65.0
LOW_THRESHOLD  = 40.0


def generate_health_insight(
    health_score: float,
    savings_rate_score: float,
    income_consistency: float,
    expense_stability: float,
) -> str:
    """
    Generate a personalised, plain-English financial health insight.

    The insight is rule-based and prioritises the most impactful
    financial issue or achievement. Logic follows a decision tree
    ordered by the component weights in the health score formula.

    Parameters
    ----------
    health_score       : float  Overall health score [0, 100].
    savings_rate_score : float  Normalised savings rate score [0, 100].
    income_consistency : float  Normalised income consistency score [0, 100].
    expense_stability  : float  Normalised expense stability score [0, 100].

    Returns
    -------
    str  A single, human-readable insight sentence.
    """
    # ── Excellent overall ────────────────────────────────────────────────────
    if health_score >= 80:
        if savings_rate_score >= HIGH_THRESHOLD and income_consistency >= HIGH_THRESHOLD:
            return (
                "Excellent financial health! Your savings rate is strong and income "
                "is highly consistent. Consider exploring investment opportunities "
                "to grow your wealth further."
            )
        if expense_stability >= HIGH_THRESHOLD:
            return (
                "Great job maintaining stable expenses and a healthy savings rate. "
                "Keep up this disciplined spending behaviour to sustain your excellent score."
            )
        return (
            "Your financial health is excellent. Continue your current savings "
            "and spending patterns to maintain this strong position."
        )

    # ── Good overall ─────────────────────────────────────────────────────────
    if health_score >= 60:
        if savings_rate_score < LOW_THRESHOLD:
            return (
                "Your income and expense patterns are healthy, but your savings rate "
                "is below target. Try increasing your monthly savings by 5-10% to "
                "push your health score into the Excellent tier."
            )
        if income_consistency < LOW_THRESHOLD:
            return (
                "Your savings rate is good, but income is irregular. Establishing a "
                "more consistent income stream or maintaining a larger emergency fund "
                "will improve your financial resilience."
            )
        if expense_stability < LOW_THRESHOLD:
            return (
                "Your savings behaviour is on track, but expense variability is high. "
                "Budgeting monthly discretionary spend will help stabilise your expenses "
                "and improve your overall score."
            )
        return (
            "Good financial health overall. Focus on small improvements in savings "
            "and spending consistency to reach Excellent status."
        )

    # ── Fair overall ─────────────────────────────────────────────────────────
    if health_score >= 40:
        if savings_rate_score < LOW_THRESHOLD and expense_stability < LOW_THRESHOLD:
            return (
                "Both your savings rate and expense stability need improvement. "
                "Start by setting a fixed monthly savings target and tracking "
                "discretionary expenses in a budget app."
            )
        if savings_rate_score < LOW_THRESHOLD:
            return (
                "Your primary concern is a low savings rate. Focus on reducing "
                "non-essential spending to free up more money for savings each month."
            )
        if income_consistency < LOW_THRESHOLD and expense_stability < LOW_THRESHOLD:
            return (
                "Both income consistency and expense stability are concerning. "
                "Building an emergency fund of 3-6 months of expenses is your most "
                "important immediate action."
            )
        return (
            "Your financial health needs attention. Identify your top spending "
            "category and set a reduction target for next month."
        )

    # ── Poor overall ─────────────────────────────────────────────────────────
    if savings_rate_score < LOW_THRESHOLD and income_consistency < LOW_THRESHOLD:
        return (
            "Critical: Both savings rate and income consistency are very low. "
            "Seek financial counselling and prioritise building any level of "
            "regular savings, even a small fixed amount each month."
        )
    if expense_stability < LOW_THRESHOLD:
        return (
            "Urgent attention needed. Highly variable expenses are depleting your "
            "savings capacity. Create a strict monthly budget and eliminate "
            "unnecessary recurring expenditures immediately."
        )
    return (
        "Your financial health score is in the Poor range. Consult the WealthWise "
        "AI Financial Coach for a personalised recovery plan tailored to your spending patterns."
    )


# ── Apply to full dataset ─────────────────────────────────────────────────────
df['health_insight'] = df.apply(
    lambda row: generate_health_insight(
        health_score       = row['health_score'],
        savings_rate_score = row['savings_rate_score'],
        income_consistency = row['income_consistency_score_n'],
        expense_stability  = row['expense_stability_score_n'],
    ),
    axis=1,
)

print("✅ Personalised insights generated for all", len(df), "records.")
print()
insight_dist = df['health_insight'].value_counts()
print(f"  Unique insight templates used: {len(insight_dist)}")

In [ ]:
# ── Sample Insight Display ────────────────────────────────────────────────────
print("💬 Sample Personalised Insights by Health Category")
print("=" * 70)

display_cols = [
    'account_number', 'month_year',
    'health_score', 'health_category',
    'strengths', 'weaknesses', 'health_insight'
]

for cat in ['Excellent', 'Good', 'Fair', 'Poor']:
    subset = df.loc[df['health_category'] == cat, display_cols]
    if subset.empty:
        continue
    sample_row = subset.sample(1, random_state=42).iloc[0]
    badge = {'Excellent': '🟢', 'Good': '🟡', 'Fair': '🟠', 'Poor': '🔴'}.get(cat, '')
    print(f"\n{badge} [{cat.upper()}]  |  Account: {sample_row['account_number']}  "
          f"|  Month: {sample_row['month_year']}  "
          f"|  Score: {sample_row['health_score']:.1f}/100")
    print(f"  Strengths : {sample_row['strengths']}")
    print(f"  Weaknesses: {sample_row['weaknesses']}")
    print(f"  Insight   : {sample_row['health_insight']}")

---

# Section 11 — Exploratory Analysis

---

Five professional visualisations covering the full health score landscape.

In [ ]:
# =============================================================================
# SECTION 11: EXPLORATORY ANALYSIS
# =============================================================================

# ── 11.1 Health Score Distribution ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5), dpi=FIGURE_DPI)

# Histogram
ax.hist(
    df['health_score'], bins=40,
    color=BRAND['score'], alpha=0.75, edgecolor='white', linewidth=0.5,
    label='Health Score'
)

# Category boundary lines
boundaries = [(40, 'Poor / Fair', '#E67E22'), (60, 'Fair / Good', '#F39C12'),
              (80, 'Good / Excellent', '#27AE60')]
for boundary, label, color in boundaries:
    ax.axvline(boundary, color=color, linestyle='--', linewidth=2, label=f'{label} boundary ({boundary})')

# Stats lines
ax.axvline(df['health_score'].mean(),   color='#C0392B', linestyle='-',  linewidth=2,
           label=f'Mean: {df["health_score"].mean():.1f}')
ax.axvline(df['health_score'].median(), color='#2C3E50', linestyle=':',  linewidth=2,
           label=f'Median: {df["health_score"].median():.1f}')

ax.set_title(
    'Financial Health Score Distribution — WealthWise AI',
    fontsize=14, fontweight='bold', pad=14
)
ax.set_xlabel('Health Score (0 – 100)', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.legend(fontsize=10, loc='upper left')
ax.set_xlim(0, 100)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('health_score_distribution.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Chart 1 saved: health_score_distribution.png")

In [ ]:
# ── 11.2 Health Score by Category (Violin + Box) ─────────────────────────────
cat_order = [c for c in ['Poor', 'Fair', 'Good', 'Excellent']
             if c in df['health_category'].unique()]
pal       = {c: HEALTH_COLORS[c] for c in cat_order}

fig, ax = plt.subplots(figsize=(12, 5), dpi=FIGURE_DPI)

sns.violinplot(
    data=df, x='health_category', y='health_score',
    order=cat_order, palette=pal,
    inner='quartile', ax=ax, linewidth=1.2,
)

ax.set_title(
    'Health Score Distribution within Each Category',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_xlabel('Health Category', fontsize=11)
ax.set_ylabel('Health Score', fontsize=11)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('health_score_violin.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Chart 2 saved: health_score_violin.png")

In [ ]:
# ── 11.3 Component Scores vs Health Score (Scatter + Trend) ──────────────────
scatter_specs = [
    ('savings_rate_score',         'Savings Rate Score (0-100)',         BRAND['savings']),
    ('income_consistency_score_n', 'Income Consistency Score (0-100)',   BRAND['income']),
    ('expense_stability_score_n',  'Expense Stability Score (0-100)',    BRAND['balance']),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=FIGURE_DPI)
fig.suptitle(
    'Component Scores vs Financial Health Score',
    fontsize=14, fontweight='bold', y=1.02
)

for ax, (col, xlabel, color) in zip(axes, scatter_specs):
    ax.scatter(
        df[col], df['health_score'],
        c=color, alpha=0.35, s=18, edgecolors='none',
    )
    # Trend line
    z   = np.polyfit(df[col].dropna(), df['health_score'][df[col].notna()], 1)
    p   = np.poly1d(z)
    x_line = np.linspace(df[col].min(), df[col].max(), 100)
    ax.plot(x_line, p(x_line), color='#2C3E50', linewidth=2, linestyle='--',
            label=f'Trend (slope={z[0]:.2f})')

    # Correlation
    corr = df[col].corr(df['health_score'])
    ax.text(0.05, 0.92, f'r = {corr:.3f}', transform=ax.transAxes,
            fontsize=11, fontweight='bold', color='#C0392B')

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Health Score', fontsize=10)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('component_vs_health_score.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Chart 3 saved: component_vs_health_score.png")

In [ ]:
# ── 11.4 Heatmap: Component Scores by Health Category ────────────────────────
heat_data = (
    df.groupby('health_category')[
        ['savings_rate_score', 'income_consistency_score_n', 'expense_stability_score_n', 'health_score']
    ]
    .mean()
    .round(2)
    .reindex(cat_order)
)
heat_data.columns = ['Savings Rate Score', 'Income Consistency', 'Expense Stability', 'Avg Health Score']

fig, ax = plt.subplots(figsize=(10, 4), dpi=FIGURE_DPI)
sns.heatmap(
    heat_data, annot=True, fmt='.1f', cmap='RdYlGn',
    vmin=0, vmax=100, linewidths=1, linecolor='white',
    annot_kws={'size': 12, 'weight': 'bold'},
    ax=ax, cbar_kws={'shrink': 0.8, 'label': 'Score (0-100)'},
)
ax.set_title(
    'Mean Component Scores by Health Category',
    fontsize=13, fontweight='bold', pad=14
)
ax.set_ylabel('Health Category', fontsize=11)
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=20, labelsize=11)
ax.tick_params(axis='y', rotation=0, labelsize=11)

plt.tight_layout()
plt.savefig('category_component_heatmap.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Chart 4 saved: category_component_heatmap.png")

In [ ]:
# ── 11.5 Monthly Health Score Trend (if multi-month data) ────────────────────
monthly_avg = (
    df.groupby('month_year')['health_score']
    .agg(['mean', 'median', 'std'])
    .reset_index()
    .sort_values('month_year')
)

if len(monthly_avg) > 1:
    fig, ax = plt.subplots(figsize=(14, 5), dpi=FIGURE_DPI)

    ax.plot(monthly_avg['month_year'], monthly_avg['mean'],
            color=BRAND['score'], linewidth=2.5, marker='o', markersize=7,
            label='Mean Health Score')
    ax.plot(monthly_avg['month_year'], monthly_avg['median'],
            color=BRAND['dark'], linewidth=2, linestyle='--', marker='s', markersize=6,
            label='Median Health Score')

    # Confidence band (mean +/- 0.5*std)
    ax.fill_between(
        monthly_avg['month_year'],
        monthly_avg['mean'] - 0.5 * monthly_avg['std'],
        monthly_avg['mean'] + 0.5 * monthly_avg['std'],
        alpha=0.15, color=BRAND['score'], label='Mean +/- 0.5 SD'
    )

    ax.set_title(
        'Monthly Financial Health Score Trend — All Accounts',
        fontsize=13, fontweight='bold', pad=14
    )
    ax.set_xlabel('Month', fontsize=11)
    ax.set_ylabel('Health Score (0 – 100)', fontsize=11)
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig('health_score_trend.png', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    print("📊 Chart 5 saved: health_score_trend.png")
else:
    print("  ℹ️  Single-month dataset — trend chart skipped.")

---

# Section 12 — Top and Bottom Users

---

Identifying the highest and lowest performing accounts enables:
- **Top users** — Premium user recognition, upsell of investment products
- **Bottom users** — Priority intervention by AI Financial Coach, risk alerts

In [ ]:
# =============================================================================
# SECTION 12: TOP AND BOTTOM USERS
# =============================================================================

DISPLAY_COLS = [
    'account_number', 'month_year',
    'health_score', 'health_category',
    'savings_rate_score', 'income_consistency_score_n', 'expense_stability_score_n',
    'strengths', 'weaknesses', 'health_insight',
]

# ── Top 10 Healthiest Profiles ────────────────────────────────────────────────
top_10 = (
    df.nlargest(10, 'health_score')[DISPLAY_COLS]
    .reset_index(drop=True)
)
top_10.index = top_10.index + 1   # Rank starts at 1

print("🏆 Top 10 Healthiest Financial Profiles")
print("=" * 65)
display(
    top_10.style
    .background_gradient(cmap='Greens', subset=['health_score'])
    .set_caption('Top 10 Accounts — Highest Financial Health Score')
)

In [ ]:
# ── Bottom 10 Profiles (Needing Improvement) ──────────────────────────────────
bottom_10 = (
    df.nsmallest(10, 'health_score')[DISPLAY_COLS]
    .reset_index(drop=True)
)
bottom_10.index = bottom_10.index + 1

print("⚠️  Bottom 10 Profiles — Immediate Attention Required")
print("=" * 65)
display(
    bottom_10.style
    .background_gradient(cmap='Reds_r', subset=['health_score'])
    .set_caption('Bottom 10 Accounts — Lowest Financial Health Score')
)

In [ ]:
# ── Top vs Bottom: Visual Comparison ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=FIGURE_DPI)
fig.suptitle(
    'Top 10 vs Bottom 10 — Financial Health Score Comparison',
    fontsize=14, fontweight='bold', y=1.02
)

for ax, (data, title, color, n_label) in zip(
    axes,
    [
        (top_10,    'Top 10 Healthiest Accounts',       '#27AE60', 'top'),
        (bottom_10, 'Bottom 10 — Needs Improvement',   '#C0392B', 'bottom'),
    ]
):
    y_labels = [
        f"{acc} | {my}" for acc, my in
        zip(data['account_number'], data['month_year'])
    ]
    scores = data['health_score'].values

    bars = ax.barh(
        range(len(scores)), scores,
        color=color, alpha=0.80, edgecolor='white', linewidth=0.8
    )
    for bar, score in zip(bars, scores):
        ax.text(
            bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{score:.1f}', va='center', ha='left', fontsize=10, fontweight='bold'
        )

    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, 115)
    ax.set_xlabel('Health Score', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=12)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('top_bottom_users.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print("📊 Chart saved: top_bottom_users.png")

---

# Section 13 — Export Dataset

---

We export the final enriched dataset in both CSV and Excel formats.  
The Excel file includes a formatted summary sheet alongside the main data sheet.

In [ ]:
# =============================================================================
# SECTION 13: EXPORT DATASET
# =============================================================================

# ── Define final output columns ───────────────────────────────────────────────
CORE_COLS = [
    # Identifiers
    'account_number',
    'month_year',
]

SCORE_COLS = [
    # Computed scores
    'health_score',
    'health_category',
    'savings_rate_score',
    'income_consistency_score_n',
    'expense_stability_score_n',
    'strengths',
    'weaknesses',
    'health_insight',
]

METRIC_COLS = [
    # Key financial metrics from feature engineering
    'monthly_income',
    'monthly_expense',
    'savings',
    'savings_rate',
    'average_balance',
    'transaction_count',
    'income_consistency_score',
    'expense_stability_score',
    'spending_to_income_ratio',
    'cash_withdrawal_frequency',
    'salary_dependency_ratio',
]

# Optional metadata columns (include if present)
META_COLS = ['account_holder', 'bank_name', 'account_type']
meta_present = [c for c in META_COLS if c in df.columns]

# Flag columns
FLAG_COLS = [c for c in ['high_spending_flag', 'high_cash_withdrawal_flag', 'low_savings_flag']
             if c in df.columns]

ALL_OUTPUT_COLS = (
    CORE_COLS + meta_present + SCORE_COLS + METRIC_COLS + FLAG_COLS
)
# Keep only columns that exist
ALL_OUTPUT_COLS = [c for c in ALL_OUTPUT_COLS if c in df.columns]

export_df = df[ALL_OUTPUT_COLS].copy()

print(f"Export dataset shape: {export_df.shape[0]:,} rows x {export_df.shape[1]} columns")
print(f"Output columns ({len(ALL_OUTPUT_COLS)}):")
for i, col in enumerate(ALL_OUTPUT_COLS, 1):
    print(f"  {i:02d}. {col}")

In [ ]:
# ── Export: CSV ───────────────────────────────────────────────────────────────
csv_path = 'health_score_dataset.csv'
export_df.to_csv(csv_path, index=False)
csv_size_kb = os.path.getsize(csv_path) / 1024
print(f"✅ CSV exported  -> {csv_path}  ({csv_size_kb:.1f} KB)")


# ── Export: Excel (multi-sheet) ───────────────────────────────────────────────
xlsx_path = 'health_score_dataset.xlsx'

try:
    with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:

        # Sheet 1: Full data
        export_df.to_excel(writer, sheet_name='Health_Score_Data', index=False)

        # Sheet 2: Summary statistics
        summary = df.groupby('health_category').agg(
            Records        = ('health_score', 'count'),
            Min_Score      = ('health_score', 'min'),
            Mean_Score     = ('health_score', 'mean'),
            Median_Score   = ('health_score', 'median'),
            Max_Score      = ('health_score', 'max'),
            Mean_Savings   = ('savings_rate_score', 'mean'),
            Mean_Income    = ('income_consistency_score_n', 'mean'),
            Mean_Expense   = ('expense_stability_score_n', 'mean'),
        ).round(2).reindex(['Poor', 'Fair', 'Good', 'Excellent'])
        summary.to_excel(writer, sheet_name='Category_Summary')

        # Sheet 3: Top 10
        top_10[CORE_COLS + list(set(SCORE_COLS) & set(top_10.columns))].to_excel(
            writer, sheet_name='Top_10_Accounts', index=True
        )

        # Sheet 4: Bottom 10
        bottom_10[CORE_COLS + list(set(SCORE_COLS) & set(bottom_10.columns))].to_excel(
            writer, sheet_name='Bottom_10_Accounts', index=True
        )

    xlsx_size_kb = os.path.getsize(xlsx_path) / 1024
    print(f"✅ Excel exported -> {xlsx_path}  ({xlsx_size_kb:.1f} KB)")
    print(f"   Sheets: Health_Score_Data | Category_Summary | Top_10_Accounts | Bottom_10_Accounts")

except ImportError:
    print("⚠️  openpyxl not installed. Install with: pip install openpyxl")
    print("   CSV export is available at:", csv_path)

print()
print("📎 Exported dataset preview:")
display(export_df.head(5))

In [ ]:
# ── Final Dataset Quality Check ───────────────────────────────────────────────
print("🔍 Export Quality Verification")
print("=" * 55)

checks = [
    ('health_score between 0-100',     export_df['health_score'].between(0, 100).all()),
    ('health_category not null',       export_df['health_category'].notna().all()),
    ('strengths not null',             export_df['strengths'].notna().all()),
    ('weaknesses not null',            export_df['weaknesses'].notna().all()),
    ('health_insight not null',        export_df['health_insight'].notna().all()),
    ('no duplicate rows',              not export_df.duplicated().any()),
    ('account_number present',         'account_number' in export_df.columns),
    ('month_year present',             'month_year' in export_df.columns),
]

all_passed = True
for description, result in checks:
    icon = "✅" if result else "❌"
    print(f"  {icon}  {description}")
    if not result:
        all_passed = False

print()
if all_passed:
    print("  ✅ ALL QUALITY CHECKS PASSED. Dataset is ready for production.")
else:
    print("  ❌ Some checks failed. Review the issues above before deployment.")

---

# Section 14 — Business Interpretation

---

## 🏢 How the Financial Health Score Powers WealthWise AI

### End-to-End Platform Workflow

```
Raw Bank Statement (PDF / JSON)
         |
         v
[Notebook 01] Data Cleaning
  -> transactions_clean.csv
         |
         v
[Notebook 02] Feature Engineering
  -> features_dataset.csv
     (37 financial features per account-month)
         |
         +-----------------------------+
         |                             |
         v                             v
[Notebook 03]                 [Notebook 04]  <-- YOU ARE HERE
Risk Profile Model            Financial Health Score
-> Conservative /             -> Score: 0-100
   Moderate /                 -> Category: Poor / Fair /
   Aggressive                             Good / Excellent
         |                             |
         +------------+----------------+
                      |
                      v
[Notebook 05] Portfolio Recommendation Engine
  Uses: risk_profile + health_score + health_category
  Output: Personalised investment recommendations
  Examples:
    - Conservative + Excellent -> FD ladder + Debt MFs
    - Aggressive   + Poor      -> Financial recovery plan first
    - Moderate     + Good      -> Balanced equity + debt MF SIPs
                      |
                      v
[Notebook 06] AI Financial Coach (Gemini-powered)
  Uses: health_score + health_insight + strengths + weaknesses
  Provides: Conversational, contextual financial guidance
  Examples:
    - "Your health score improved 8 points this month!"
    - "Focus area: reducing cash withdrawals (-23% target)"
                      |
                      v
[FastAPI Backend]
  Endpoint: POST /api/health-score
  Input:    features dict (from feature engineering)
  Output:   health_score, health_category, insights
                      |
                      v
[WealthWise AI Dashboard]
  - Score gauge: 72 / 100  [GOOD]
  - Trend line: month-over-month history
  - Strengths panel: Strong Savings Rate
  - Improvement tips: derived from health_insight
  - Badge: Gold / Silver / Bronze wellness tier
```

---

### 🚀 FastAPI Integration — Production Code Pattern

```python
# app/services/health_score_service.py

class HealthScoreService:
    """
    WealthWise AI Financial Health Scoring Service.
    Importable by FastAPI for real-time scoring endpoints.
    """

    WEIGHTS = {
        'savings_rate_score':         0.50,
        'income_consistency_score_n': 0.25,
        'expense_stability_score_n':  0.25,
    }

    def score(self, features: dict) -> dict:
        """
        Compute health score from a feature dict.

        Parameters
        ----------
        features : dict  Keys must include normalised score components.

        Returns
        -------
        dict with health_score, health_category, strengths,
             weaknesses, health_insight.
        """
        score = sum(
            w * features.get(c, 0.0)
            for c, w in self.WEIGHTS.items()
        )
        score = float(np.clip(score, 0.0, 100.0))

        return {
            'health_score':    round(score, 2),
            'health_category': assign_health_category(score),
            'strengths':       identify_strengths(features, STRENGTH_RULES),
            'weaknesses':      identify_weaknesses(features, WEAKNESS_RULES),
            'health_insight':  generate_health_insight(
                score,
                features.get('savings_rate_score', 0),
                features.get('income_consistency_score_n', 0),
                features.get('expense_stability_score_n', 0),
            ),
        }


# FastAPI endpoint:
# @router.post('/api/health-score')
# async def compute_health_score(features: FeatureInput):
#     service = HealthScoreService()
#     return service.score(features.dict())
```

---

### 📊 How Users Benefit from the Health Score

| User Interaction | Health Score Role |
|-----------------|-------------------|
| Opens WealthWise AI dashboard | Sees their current score badge (e.g., **72 / 100 🟡 Good**) |
| Views monthly trend | Tracks score change month-over-month |
| Reads strength panel | Understands what they are doing right |
| Reads weakness panel | Gets a prioritised improvement target |
| Talks to AI Coach | Coach uses score + insight as context |
| Views portfolio recommendations | Recommendations filtered by score tier |
| Receives alerts | Push notification if score drops > 10 points |

---

## ✅ Notebook 04 Summary

| Section | Task | Status |
|---------|------|--------|
| 1  | Platform overview and methodology | ✅ Complete |
| 2  | Library imports | ✅ Complete |
| 3  | Dataset loading (with fallback) | ✅ Complete |
| 4  | Comprehensive data validation | ✅ Complete |
| 5  | Component normalisation (domain-capped) | ✅ Complete |
| 6  | Weighted health score computation | ✅ Complete |
| 7  | Wellness category classification | ✅ Complete |
| 8  | Automated strength identification | ✅ Complete |
| 9  | Automated weakness identification | ✅ Complete |
| 10 | Rule-based personalised insights | ✅ Complete |
| 11 | 5-chart exploratory analysis suite | ✅ Complete |
| 12 | Top 10 / Bottom 10 user analysis | ✅ Complete |
| 13 | CSV + Excel export (4 sheets) | ✅ Complete |
| 14 | Business interpretation + FastAPI pattern | ✅ Complete |

> **Next Step:** Run `05_portfolio_recommendation.ipynb` to build personalised investment recommendations using both the Risk Profile (Notebook 03) and Financial Health Score (this notebook).

In [ ]:
# =============================================================================
# FINAL SUMMARY — Output Verification
# =============================================================================

print("=" * 65)
print("  WealthWise AI -- Financial Health Score: Final Summary")
print("=" * 65)

print("\n  Dataset")
print(f"    Input records  : {len(df):,} rows")
print(f"    Unique accounts: {df['account_number'].nunique()}")
print(f"    Unique months  : {df['month_year'].nunique()}")

print("\n  Scoring")
print(f"    Weight - Savings Rate Score        : {SCORE_WEIGHTS['savings_rate_score']*100:.0f}%")
print(f"    Weight - Income Consistency Score  : {SCORE_WEIGHTS['income_consistency_score_n']*100:.0f}%")
print(f"    Weight - Expense Stability Score   : {SCORE_WEIGHTS['expense_stability_score_n']*100:.0f}%")

print("\n  Health Score Statistics")
print(f"    Min    : {df['health_score'].min():.2f}")
print(f"    Max    : {df['health_score'].max():.2f}")
print(f"    Mean   : {df['health_score'].mean():.2f}")
print(f"    Median : {df['health_score'].median():.2f}")

print("\n  Category Breakdown")
for cat in ['Excellent', 'Good', 'Fair', 'Poor']:
    cnt = (df['health_category'] == cat).sum()
    pct = cnt / len(df) * 100
    bar = "|" * int(pct / 5)
    print(f"    {cat:<12s}: {cnt:>5,} ({pct:>5.1f}%)  {bar}")

print("\n  Output Files")
for fname, desc in [
    ('health_score_dataset.csv',  'Full scored dataset (CSV)'),
    ('health_score_dataset.xlsx', 'Full scored dataset (Excel, 4 sheets)'),
]:
    exists = os.path.exists(fname)
    icon   = "✅" if exists else "❌"
    size   = f"{os.path.getsize(fname)/1024:.1f} KB" if exists else "Not found"
    print(f"    {icon} {fname:<40s} {size:>10s}  {desc}")

print("\n  Charts Generated")
charts = [
    'component_distributions.png',
    'health_category_distribution.png',
    'health_score_distribution.png',
    'health_score_violin.png',
    'component_vs_health_score.png',
    'category_component_heatmap.png',
    'strengths_weaknesses.png',
    'top_bottom_users.png',
]
for chart in charts:
    icon = "✅" if os.path.exists(chart) else "—"
    print(f"    {icon} {chart}")

print("\n  Next Step: 05_portfolio_recommendation.ipynb")
print("=" * 65)